# AI Project Planning & Risk Forecasting App (Construction Dataset Training)

This notebook trains a supervised classifier to predict `Risk_Level` from construction task features using `construction_dataset.csv`.


## 0. Environment Setup


In [1]:
# Installs required libraries in the active Jupyter kernel interpreter.
import importlib.util
import subprocess
import sys

required = {
    'numpy': 'numpy>=2.0,<3.0',
    'pandas': 'pandas>=2.2,<3.0',
    'plotly': 'plotly>=5.24,<6.0',
    'sklearn': 'scikit-learn>=1.5,<2.0',
    'joblib': 'joblib>=1.4,<2.0',
}

missing_specs = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]

if missing_specs:
    print('Installing missing packages:', ', '.join(missing_specs))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_specs])
    print('Installation complete. You can now run the imports cell.')
else:
    print('All required packages are already installed for this kernel.')


All required packages are already installed for this kernel.


## 1. Imports


In [2]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from joblib import dump


## 2. Dataset Path Configuration


In [3]:
# The notebook first checks the original attached location, then the project-local copy.
DATASET_CANDIDATES = [
    Path('/Users/santiagomolina/Documents/Neue Fische Capstone Project/construction_dataset.csv'),
    Path('data/construction_dataset.csv'),
]

DATASET_PATH = next((p for p in DATASET_CANDIDATES if p.exists()), None)
if DATASET_PATH is None:
    newline = chr(10)
    checked = newline.join(str(p) for p in DATASET_CANDIDATES)
    raise FileNotFoundError(
        'construction_dataset.csv not found. Checked paths:' + newline + checked
    )

print('Using dataset:', DATASET_PATH.resolve())


Using dataset: /Users/santiagomolina/Documents/Neue Fische Capstone Project/construction_dataset.csv


## 3. Load Data


In [4]:
df = pd.read_csv(DATASET_PATH)

print('Shape:', df.shape)
print('Columns:', list(df.columns))
print()
print('Dtypes:')
display(df.dtypes.to_frame('dtype'))
print()
print('Missing values per column:')
display(df.isna().sum().to_frame('missing_count'))

df.head()


Shape: (1300, 10)
Columns: ['Task_ID', 'Task_Duration_Days', 'Labor_Required', 'Equipment_Units', 'Material_Cost_USD', 'Start_Constraint', 'Risk_Level', 'Resource_Constraint_Score', 'Site_Constraint_Score', 'Dependency_Count']

Dtypes:


,dtype
Task_ID,object
Task_Duration_Days,int64
Labor_Required,int64
Equipment_Units,int64
Material_Cost_USD,float64
Start_Constraint,int64
Risk_Level,object
Resource_Constraint_Score,float64
Site_Constraint_Score,float64
Dependency_Count,int64



Missing values per column:


,missing_count
Task_ID,0
Task_Duration_Days,0
Labor_Required,0
Equipment_Units,0
Material_Cost_USD,0
Start_Constraint,0
Risk_Level,0
Resource_Constraint_Score,0
Site_Constraint_Score,0
Dependency_Count,0


,Task_ID,Task_Duration_Days,Labor_Required,Equipment_Units,Material_Cost_USD,Start_Constraint,Risk_Level,Resource_Constraint_Score,Site_Constraint_Score,Dependency_Count
0,T1,52,14,6,16789.73,0,Medium,0.41,0.59,4
1,T2,15,2,2,16885.80,5,Low,0.75,0.17,3
2,T3,72,11,1,7978.70,22,Low,0.96,0.41,1
3,T4,61,1,5,19379.02,18,Low,0.41,0.67,4
4,T5,21,19,5,66757.72,22,Low,0.85,0.63,3


## 4. Quick Target Distribution


In [5]:
import sys
!{sys.executable} -m pip install --upgrade nbformat

In [6]:
target_col = 'Risk_Level'

if target_col not in df.columns:
    raise KeyError(f'Missing target column: {target_col}')

risk_counts = df[target_col].value_counts()
risk_share = df[target_col].value_counts(normalize=True).mul(100).round(2)

summary_df = pd.DataFrame({'count': risk_counts, 'share_pct': risk_share})
display(summary_df)

fig_target = px.bar(
    summary_df.reset_index(),
    x='Risk_Level',
    y='count',
    title='Risk Level Distribution',
    text='count',
)
fig_target.update_traces(textposition='outside')
fig_target.show(renderer="iframe")


,count,share_pct
Risk_Level,,
Low,660,50.77
Medium,377,29.00
High,263,20.23


## 5. Feature Matrix and Train/Test Split


In [7]:
ID_COL = 'Task_ID'
TARGET_COL = 'Risk_Level'

feature_cols = [c for c in df.columns if c not in {ID_COL, TARGET_COL}]
X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('Feature columns used for training:')
print(feature_cols)
print()
print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)
print('Train distribution:')
print(y_train.value_counts(normalize=True).round(3))


Feature columns used for training:
['Task_Duration_Days', 'Labor_Required', 'Equipment_Units', 'Material_Cost_USD', 'Start_Constraint', 'Resource_Constraint_Score', 'Site_Constraint_Score', 'Dependency_Count']

Train shape: (1040, 8) | Test shape: (260, 8)
Train distribution:
Risk_Level
Low       0.508
Medium    0.290
High      0.202
Name: proportion, dtype: float64


## 6. Baseline Model (Dummy Classifier)


In [8]:
baseline_model = DummyClassifier(strategy='most_frequent')
baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_f1_macro = f1_score(y_test, baseline_pred, average='macro')

print(f'Baseline accuracy     : {baseline_acc:.4f}')
print(f'Baseline macro F1     : {baseline_f1_macro:.4f}')


Baseline accuracy     : 0.5077
Baseline macro F1     : 0.2245


## 7. Train Final Model (Random Forest)


In [9]:
model = Pipeline(
    steps=[
        (
            'classifier',
            RandomForestClassifier(
                n_estimators=400,
                max_depth=None,
                min_samples_leaf=2,
                class_weight='balanced',
                random_state=42,
                n_jobs=-1,
            ),
        )
    ]
)

model.fit(X_train, y_train)
print('Final model trained.')


Final model trained.


## 8. Evaluate Final Model


In [10]:
y_pred = model.predict(X_test)
final_acc = accuracy_score(y_test, y_pred)
final_f1_macro = f1_score(y_test, y_pred, average='macro')

print(f'Final accuracy        : {final_acc:.4f}')
print(f'Final macro F1        : {final_f1_macro:.4f}')
print(f'Accuracy improvement  : {final_acc - baseline_acc:+.4f}')
print(f'Macro F1 improvement  : {final_f1_macro - baseline_f1_macro:+.4f}')
print()
print('Classification report:')
print(classification_report(y_test, y_pred))


Final accuracy        : 0.4346
Final macro F1        : 0.2643
Accuracy improvement  : -0.0731
Macro F1 improvement  : +0.0398

Classification report:
              precision    recall  f1-score   support

        High       0.08      0.02      0.03        53
         Low       0.49      0.78      0.60       132
      Medium       0.23      0.12      0.16        75

    accuracy                           0.43       260
   macro avg       0.27      0.31      0.26       260
weighted avg       0.33      0.43      0.36       260



In [11]:
# Keep class order business-readable.
class_order = ['Low', 'Medium', 'High']
class_order = [c for c in class_order if c in set(y)]

cm = confusion_matrix(y_test, y_pred, labels=class_order)
cm_df = pd.DataFrame(
    cm,
    index=[f'true_{c}' for c in class_order],
    columns=[f'pred_{c}' for c in class_order],
)

display(cm_df)

fig_cm = px.imshow(
    cm_df,
    text_auto=True,
    title='Confusion Matrix (Test Set)',
    color_continuous_scale='Blues',
)
fig_cm.show()


,pred_Low,pred_Medium,pred_High
true_Low,103,20,9
true_Medium,64,9,2
true_High,42,10,1


## 9. Cross-Validation Check


In [12]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

try:
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
except Exception as exc:
    print(f'Parallel cross-validation failed ({type(exc).__name__}: {exc}). Retrying with n_jobs=1...')
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring='f1_macro', n_jobs=1)

print('CV macro-F1 scores:', np.round(cv_scores, 4))
print(f'CV macro-F1 mean   : {cv_scores.mean():.4f}')
print(f'CV macro-F1 std    : {cv_scores.std():.4f}')


CV macro-F1 scores: [0.274  0.2753 0.2696 0.28   0.2694]
CV macro-F1 mean   : 0.2737
CV macro-F1 std    : 0.0039


## 10. Feature Importance


In [13]:
rf = model.named_steps['classifier']

importance_df = pd.DataFrame(
    {
        'feature': feature_cols,
        'importance': rf.feature_importances_,
    }
).sort_values('importance', ascending=False)

importance_df


,feature,importance
3,Material_Cost_USD,0.162947
5,Resource_Constraint_Score,0.146422
6,Site_Constraint_Score,0.143138
0,Task_Duration_Days,0.142081
4,Start_Constraint,0.129175
1,Labor_Required,0.117969
2,Equipment_Units,0.090184
7,Dependency_Count,0.068083


In [14]:
fig_importance = px.bar(
    importance_df,
    x='importance',
    y='feature',
    orientation='h',
    title='Feature Importance (Random Forest)',
)
fig_importance.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_importance.show()


## 11. Save Trained Model and Metrics


In [15]:
if 'model' not in globals():
    raise RuntimeError('`model` not found. Run section 7 (Train Final Model) first.')

if 'cv_scores' not in globals():
    print('`cv_scores` not found. Running fallback cross-validation in this cell...')
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring='f1_macro', n_jobs=1)

if 'importance_df' not in globals():
    print('`importance_df` not found. Computing feature importances in this cell...')
    rf = model.named_steps['classifier']
    importance_df = pd.DataFrame(
        {
            'feature': feature_cols,
            'importance': rf.feature_importances_,
        }
    ).sort_values('importance', ascending=False)

models_dir = Path('models')
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / 'risk_classifier.joblib'
dump(model, model_path)

metrics_payload = {
    'dataset_path': str(DATASET_PATH),
    'n_rows': int(df.shape[0]),
    'n_features': int(len(feature_cols)),
    'baseline_accuracy': float(baseline_acc),
    'baseline_macro_f1': float(baseline_f1_macro),
    'final_accuracy': float(final_acc),
    'final_macro_f1': float(final_f1_macro),
    'cv_macro_f1_mean': float(cv_scores.mean()),
    'cv_macro_f1_std': float(cv_scores.std()),
    'feature_importance': importance_df.to_dict(orient='records'),
}

metrics_path = models_dir / 'risk_model_metrics.json'
metrics_path.write_text(json.dumps(metrics_payload, indent=2))

print('Saved model   :', model_path.resolve())
print('Saved metrics :', metrics_path.resolve())


Saved model   : /Users/santiagomolina/aipm-1711/ai-powered-project-planning-risk-forecasting-app/models/risk_classifier.joblib
Saved metrics : /Users/santiagomolina/aipm-1711/ai-powered-project-planning-risk-forecasting-app/models/risk_model_metrics.json


## 12. Single-Row Inference Example


In [16]:
sample = X_test.iloc[[0]].copy()
true_label = y_test.iloc[0]

pred_label = model.predict(sample)[0]
pred_prob = model.predict_proba(sample)[0]
prob_df = pd.DataFrame(
    {
        'Risk_Level': model.named_steps['classifier'].classes_,
        'Probability': np.round(pred_prob, 4),
    }
).sort_values('Probability', ascending=False)

print('True label     :', true_label)
print('Predicted label:', pred_label)
display(sample)
prob_df


True label     : Low
Predicted label: High


,Task_Duration_Days,Labor_Required,Equipment_Units,Material_Cost_USD,Start_Constraint,Resource_Constraint_Score,Site_Constraint_Score,Dependency_Count
680,66,11,1,61666.79,14,0.96,0.32,0


,Risk_Level,Probability
0,High,0.3391
2,Medium,0.3351
1,Low,0.3258


## 13. Notes

- Model target is `Risk_Level` (`Low`, `Medium`, `High`).
- `Task_ID` is excluded from features because it is an identifier, not a predictive signal.
- For production, use this notebook output to build a dedicated training script and model versioning pipeline.
